# CaImAn results report (motion + components)

This notebook loads a CaImAn `cnm.save(...).hdf5` results file and generates a compact set of figures you can drop into a presentation:

- Motion correction diagnostics (if shifts are stored in the results `.hdf5`; otherwise it will guide you to the memmap / MC outputs)
- Summary images (mean, max, correlation)
- Spatial footprints montage + contours overlay
- Example temporal traces with deconvolved spikes

**Edit only the paths in the next cell and run top to bottom.**


In [ ]:
# --- USER INPUTS (edit these) ---
import os

CAIMAN_HDF5 = r"/media/toor/SeagateClean/Mouse946/Round1/Linear/2026-02-05T16_54_25/caiman_final/caiman_results.hdf5"

# Optional: if you want before/after frames, provide your original raw movie path
# (tif / avi / hdf5 / memmap). If empty, the notebook will still run.
RAW_MOVIE = r""  # e.g. r"/path/to/raw.tif"

# Output folder for presentation-ready PNGs
OUTDIR = "figs_caiman_report"
os.makedirs(OUTDIR, exist_ok=True)

# A small frame window to keep things fast
N_FRAMES_PREVIEW = 1000      # how many frames to sample for summary images
FRAME_STEP = 5               # subsampling step for preview


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# CaImAn imports (assumes caiman is installed in this environment)
import caiman as cm
from caiman.source_extraction.cnmf.cnmf import load_CNMF

# Optional visualization helpers (will fallback if missing)
try:
    from caiman.utils.visualization import plot_contours
    _HAS_PLOT_CONTOURS = True
except Exception:
    _HAS_PLOT_CONTOURS = False

import h5py


In [ ]:
def describe_hdf5(path, max_items=200):
    # Lightweight peek inside an HDF5 file (names + shapes).
    out = []
    with h5py.File(path, "r") as f:
        def walk(name, obj):
            if isinstance(obj, h5py.Dataset):
                out.append((name, obj.shape, str(obj.dtype)))
        f.visititems(walk)

    for name, shape, dtype in out[:max_items]:
        print(f"{name:70s}  shape={shape}  dtype={dtype}")
    if len(out) > max_items:
        print(f"... showing first {max_items} datasets (total {len(out)})")

print("CaImAn results file:", CAIMAN_HDF5)
if not os.path.exists(CAIMAN_HDF5):
    raise FileNotFoundError("CAIMAN_HDF5 path not found. Edit the path in the first cell.")

# Load CNMF object saved with cnm.save("...hdf5")
cnm = load_CNMF(CAIMAN_HDF5, dview=None)
print("Loaded CNMF object.")
print("Dims:", cnm.dims)
print("n_components:", cnm.estimates.A.shape[1])

# Optional: list datasets stored inside the HDF5 (useful to locate motion shifts)
print("\n--- HDF5 datasets (first items) ---")
describe_hdf5(CAIMAN_HDF5, max_items=120)


In [ ]:
# -------------------------------------------------
# Reconstruct denoised movie from CNMF results
# -------------------------------------------------

A = cnm.estimates.A          # (pixels x K)
C = cnm.estimates.C          # (K x T)
b = cnm.estimates.b          # (pixels x nb)
f = cnm.estimates.f          # (nb x T)

d1, d2 = cnm.dims
T = C.shape[1]

print("Reconstructing denoised movie...")

# Low-rank reconstruction
Y_rec = (A @ C + b @ f)      # (pixels x T)

# Reshape to movie (T, d1, d2)
Y_rec = np.array(Y_rec.T).reshape((T, d1, d2), order="F")

print("Reconstructed movie shape:", Y_rec.shape)


In [ ]:
mean_img = Y_rec.mean(axis=0)
max_img  = Y_rec.max(axis=0)

plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(mean_img, cmap='gray')
plt.title("Mean (reconstructed)")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(max_img, cmap='gray')
plt.title("Max (reconstructed)")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
def savefig(name):
    path = os.path.join(OUTDIR, name)
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved:", path)

# Summary images from corrected movie
Y = np.array(mov_corr)  # (T, d1, d2)
mean_img = Y.mean(axis=0)
max_img  = Y.max(axis=0)
corr_img, pnr_img = cm.summary_images.correlation_pnr(Y, gSig=None)

plt.figure(figsize=(12,3))
plt.subplot(1,4,1); plt.imshow(mean_img, cmap="gray"); plt.title("Mean (corrected)"); plt.axis("off")
plt.subplot(1,4,2); plt.imshow(max_img,  cmap="gray"); plt.title("Max (corrected)");  plt.axis("off")
plt.subplot(1,4,3); plt.imshow(corr_img, cmap="gray"); plt.title("Correlation");      plt.axis("off")
plt.subplot(1,4,4); plt.imshow(pnr_img,  cmap="gray"); plt.title("PNR");              plt.axis("off")
plt.tight_layout()
savefig("01_summary_images.png")
plt.show()

# Optional: show raw vs corrected mean if raw is provided
if mov_raw is not None:
    Y0 = np.array(mov_raw)
    mean_raw = Y0.mean(axis=0)
    plt.figure(figsize=(8,3))
    plt.subplot(1,2,1); plt.imshow(mean_raw, cmap="gray"); plt.title("Mean (raw)"); plt.axis("off")
    plt.subplot(1,2,2); plt.imshow(mean_img, cmap="gray"); plt.title("Mean (corrected)"); plt.axis("off")
    plt.tight_layout()
    savefig("02_mean_raw_vs_corrected.png")
    plt.show()


In [ ]:
def extract_motion_shifts_from_hdf5(path):
    # Tries to find motion correction shifts stored in the CaImAn results HDF5.
    shifts = {}
    with h5py.File(path, "r") as f:
        candidates = []
        def walk(name, obj):
            if isinstance(obj, h5py.Dataset):
                lname = name.lower()
                if "shift" in lname and obj.size > 0 and obj.ndim in (1,2):
                    candidates.append(name)
        f.visititems(walk)

        preferred = [n for n in candidates if any(k in n.lower() for k in ["x_shifts","y_shifts","shifts_rig","shifts"])]
        for n in preferred + candidates:
            data = np.array(f[n])
            lname = n.lower()
            if data.ndim == 1 and ("x_shifts" in lname or lname.endswith("/x")):
                shifts["x"] = data
            elif data.ndim == 1 and ("y_shifts" in lname or lname.endswith("/y")):
                shifts["y"] = data
            elif data.ndim == 2 and data.shape[1] == 2:
                shifts["xy_pair"] = data
                shifts["_source"] = n
                break
    return shifts

sh = extract_motion_shifts_from_hdf5(CAIMAN_HDF5)

if len(sh) == 0:
    print(
        "No motion shifts found inside this results HDF5.\n"
        "This is common if you saved only the CNMF object.\n\n"
        "If you want motion-correction plots, save the MotionCorrect object during your pipeline,\n"
        "or keep the raw movie and show raw vs corrected mean / sample frames."
    )
else:
    plt.figure(figsize=(10,3))
    if "x" in sh and "y" in sh:
        plt.plot(sh["x"], label="x shift")
        plt.plot(sh["y"], label="y shift")
        plt.legend()
        plt.title("Motion correction shifts")
    elif "xy_pair" in sh:
        xy = sh["xy_pair"]
        plt.plot(xy[:,0], label="shift dim0")
        plt.plot(xy[:,1], label="shift dim1")
        plt.legend()
        plt.title(f"Motion correction shifts (from {sh.get('_source','HDF5')})")
    plt.xlabel("Frame")
    plt.ylabel("Shift (pixels)")
    plt.tight_layout()
    savefig("03_motion_shifts.png")
    plt.show()


In [ ]:
A = cnm.estimates.A                 # sparse (d x K)
C = cnm.estimates.C                 # (K x T)
S = getattr(cnm.estimates, "S", None)

dims = cnm.dims
d1, d2 = dims
K = A.shape[1]
print("K components:", K)

def footprints_montage(A, dims, k_list, ncols=10):
    d1, d2 = dims
    imgs = []
    for k in k_list:
        ak = A[:, k].toarray().reshape((d1, d2), order="F")
        imgs.append(ak)
    imgs = np.array(imgs)
    imgs = imgs / (imgs.max(axis=(1,2), keepdims=True) + 1e-9)

    n = len(k_list)
    nrows = int(np.ceil(n / ncols))
    canvas = np.zeros((nrows*d1, ncols*d2), dtype=float)
    for i in range(n):
        r, c = divmod(i, ncols)
        canvas[r*d1:(r+1)*d1, c*d2:(c+1)*d2] = imgs[i]
    return canvas

n_show = min(50, K)
k_list = list(range(n_show))
mont = footprints_montage(A, dims, k_list, ncols=10)

plt.figure(figsize=(12, 6))
plt.imshow(mont, cmap="gray")
plt.title(f"Spatial footprints montage (first {n_show} components)")
plt.axis("off")
plt.tight_layout()
savefig("04_spatial_footprints_montage.png")
plt.show()

if _HAS_PLOT_CONTOURS:
    plt.figure(figsize=(6,6))
    plt.imshow(corr_img, cmap="gray")
    plot_contours(A, corr_img, thr=0.9, display_numbers=False)
    plt.title("Component contours over correlation image")
    plt.axis("off")
    plt.tight_layout()
    savefig("05_contours_on_corr.png")
    plt.show()
else:
    print("plot_contours not available in this CaImAn install. Skipping contour overlay.")


In [ ]:
# Pick a few example components with high activity for demo traces
energy = C.var(axis=1)
idx = np.argsort(-energy)[:8]
print("Example components:", idx.tolist())

def plot_trace(k, t0=0, t1=2000):
    t1 = min(t1, C.shape[1])
    plt.figure(figsize=(12,3))
    plt.plot(C[k, t0:t1], label="C (denoised)")
    if S is not None:
        s = S[k, t0:t1]
        if np.max(s) > 0:
            plt.plot((s/np.max(s))*np.max(C[k, t0:t1]), label="S (deconvolved, scaled)")
    plt.title(f"Component {k}: trace + deconvolved spikes")
    plt.xlabel("Frame")
    plt.ylabel("a.u.")
    plt.legend(loc="upper right")
    plt.tight_layout()
    savefig(f"06_trace_component_{k}.png")
    plt.show()

for k in idx:
    plot_trace(int(k), t0=0, t1=min(3000, C.shape[1]))


In [ ]:
# Optional: basic QC metrics (if present)
qc_fields = ["SNR_comp", "r_values", "cnn_preds"]
for f in qc_fields:
    if hasattr(cnm.estimates, f):
        v = np.array(getattr(cnm.estimates, f)).ravel()
        print(f"{f}: n={len(v)}  median={np.nanmedian(v):.3g}  min={np.nanmin(v):.3g}  max={np.nanmax(v):.3g}")

for f in qc_fields:
    if hasattr(cnm.estimates, f):
        v = np.array(getattr(cnm.estimates, f)).ravel()
        plt.figure(figsize=(5,3))
        plt.hist(v[~np.isnan(v)], bins=30)
        plt.title(f"Histogram: {f}")
        plt.tight_layout()
        savefig(f"07_hist_{f}.png")
        plt.show()


## What to show in a “CaImAn pipeline” slide

1. Motion correction (NoRMCorre): raw vs corrected mean, and shifts if available  
2. Source extraction (CNMF/CNMF-E): correlation image + contours  
3. Denoising + deconvolution: trace plus inferred spikes  
4. Quality control: SNR / r-values / CNN (if computed)

All figures are saved to `figs_caiman_report/`.
